In [12]:
import pandas as pd
import json

file_path = 'C:/Users/t-toluale/.vscode/Ordering_ChatBot/evaluation_results/contoso_restaurant_conversation_20250805_142040.json'
with open(file_path, 'r') as f:
    evaluation_results = json.load(f)

df = pd.DataFrame(evaluation_results)
df

,brand_name,conversation,timestamp
0,contoso_restaurant,"{'role': 'user', 'content': 'Hello'}",2025-08-05T14:20:40.244444
1,contoso_restaurant,"{'role': 'assistant', 'content': 'Hello! Welco...",2025-08-05T14:20:40.244444
2,contoso_restaurant,"{'role': 'user', 'content': 'Hello, I want to ...",2025-08-05T14:20:40.244444
3,contoso_restaurant,"{'role': 'assistant', 'content': 'That sounds ...",2025-08-05T14:20:40.244444
4,contoso_restaurant,"{'role': 'user', 'content': 'What do you recom...",2025-08-05T14:20:40.244444
5,contoso_restaurant,"{'role': 'assistant', 'content': 'We’d love to...",2025-08-05T14:20:40.244444
6,contoso_restaurant,"{'role': 'user', 'content': 'Can you put toget...",2025-08-05T14:20:40.244444
7,contoso_restaurant,"{'role': 'assistant', 'content': 'Absolutely! ...",2025-08-05T14:20:40.244444
8,contoso_restaurant,"{'role': 'user', 'content': 'Include medium si...",2025-08-05T14:20:40.244444
9,contoso_restaurant,"{'role': 'assistant', 'content': 'Got it! Here...",2025-08-05T14:20:40.244444


In [13]:
conversations = []
# Since evaluation_results is a single dictionary, access it directly
brand_name = evaluation_results['brand_name']
conversation_data = evaluation_results['conversation']

# Extract conversation pairs
for i in range(0, len(conversation_data), 2):
    if i + 1 < len(conversation_data):  
        user_msg = conversation_data[i]
        assistant_msg = conversation_data[i + 1]
        
        conversations.append({
            'brand_name': brand_name,
            'user_prompt': user_msg['content'],
            'assistant_response': assistant_msg['content']
        })

In [14]:
conversations_df = pd.DataFrame(conversations)
conversations_df.to_csv('C:/Users/t-toluale/.vscode/Ordering_ChatBot/evaluation_results/conversations_4o.csv', index=False)
conversations_df

,brand_name,user_prompt,assistant_response
0,contoso_restaurant,Hello,"Hello! Welcome to Contoso Restaurant, where ev..."
1,contoso_restaurant,"Hello, I want to order dinner for my family",That sounds wonderful! We’d be delighted to he...
2,contoso_restaurant,What do you recommend on your menu,We’d love to help you choose the perfect dinne...
3,contoso_restaurant,Can you put together a meal for 4 people?,"Absolutely! Let’s create a delicious, home-sty..."
4,contoso_restaurant,Include medium size drinks for everyone,"Got it! Here’s your updated meal for four, now..."
5,contoso_restaurant,let me have a cheeseburger. Increase the order...,Of course! Let’s adjust the order to serve six...
6,contoso_restaurant,Can I get a pretzel hamburger with unsalted fr...,Absolutely! Here’s your customized order:\n\n#...
7,contoso_restaurant,I ordered a cheeseburger with unsalted fries a...,Got it! Here’s your updated order based on you...
8,contoso_restaurant,I want a pizza with pineapple and olives as to...,We currently specialize in home-style American...
9,contoso_restaurant,Can you summarize my order?,Of course! Here’s your order summary:\n\n### Y...


In [26]:
evaluations = []

for brand_data in evaluation_results: 
    eval_data = { 'brand_name': brand_data['brand_name'], 
                 'BrandVoiceMetric': brand_data['scores']['BrandVoiceMetric'], 
                 'RelevanceMetric': brand_data['scores']['RelevanceMetric'], 
                 'TaskCompletionMetric': brand_data['scores']['TaskCompletionMetric'] } 
    evaluations.append(eval_data)

evaluation_df = pd.DataFrame(evaluations) 
evaluation_df.to_csv('C:/Users/t-toluale/.vscode/Ordering_ChatBot/evaluation_results/evaluation_scores.csv', index=False)

#Evaluation

In [8]:
import os
from azure.ai.evaluation import RelevanceEvaluator, CoherenceEvaluator, FluencyEvaluator
from dotenv import load_dotenv
from azure.ai.evaluation._model_configurations import AzureOpenAIModelConfiguration


load_dotenv()

ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
DEPLOYMENT_NAME = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")

model_config = {
    "azure_endpoint": ENDPOINT,
    "azure_deployment": DEPLOYMENT_NAME,
    "api_key": API_KEY,
}

if not model_config.get("azure_endpoint"):
	raise ValueError("Missing required Azure endpoint in model_config.")
if not model_config.get("azure_deployment"):
	raise ValueError("Missing required Azure deployment in model_config.")
if not model_config.get("api_key"):
	raise ValueError("Missing required API key in model_config.")

azure_model_config = AzureOpenAIModelConfiguration(
	azure_endpoint=str(model_config["azure_endpoint"]),
	azure_deployment=str(model_config["azure_deployment"]),
	api_key=str(model_config["api_key"]),
	api_version=str(model_config.get("api_version", "2024-12-01-preview"))
)
relevance_eval = RelevanceEvaluator(model_config=azure_model_config, threshold=3)
coherence_eval = CoherenceEvaluator(model_config=azure_model_config, threshold=3) 
fluency_eval = FluencyEvaluator(model_config=azure_model_config, threshold=3)

test_query = "What is the capital of Japan?" 
test_response = "The capital of Japan is Tokyo."

print("Testing Relevance Evaluator:") 
relevance_result = relevance_eval(query=test_query, response=test_response) 
print(f"Relevance result: {relevance_result}")

print("\nTesting Coherence Evaluator:") 
coherence_result = coherence_eval(query=test_query, response=test_response) 
print(f"Coherence result: {coherence_result}")

print("\nTesting Fluency Evaluator:") 
fluency_result = fluency_eval(query=test_query, response=test_response) 
print(f"Fluency result: {fluency_result}")
   

Testing Relevance Evaluator:
Relevance result: {'relevance': 4.0, 'gpt_relevance': 4.0, 'relevance_reason': 'The response fully and accurately answers the question with all necessary details, but does not provide any extra insights.', 'relevance_result': 'pass', 'relevance_threshold': 3}

Testing Coherence Evaluator:
Coherence result: {'coherence': 4.0, 'gpt_coherence': 4.0, 'coherence_reason': 'The response is logically organized, directly answers the question, and is easy to understand, meeting the criteria for a coherent response.', 'coherence_result': 'pass', 'coherence_threshold': 3}

Testing Fluency Evaluator:
Fluency result: {'fluency': 3.0, 'gpt_fluency': 3.0, 'fluency_reason': 'The response is clear, correct, and easily understood, but it is very simple and lacks complexity or advanced vocabulary.', 'fluency_result': 'pass', 'fluency_threshold': 3}


In [12]:
import pandas as pd
file_path = 'C:/Users/t-toluale/.vscode/Ordering_ChatBot/evaluation_results/evaluation_4'
conversations_df = pd.read_csv(file_path + '/conversations.csv')


In [15]:
import asyncio
import time

async def evaluate_conversations_multi_metrics(conversations_df, batch_size=2, delay_between_batches=60): 
    evaluation_results = [] # Add delay between API calls to respect rate limits 
    total_batches = len(conversations_df) // batch_size + (1 if len(conversations_df) % batch_size else 0)
    for batch_num in range(total_batches):
        start_idx = batch_num * batch_size
        end_idx = min((batch_num + 1) * batch_size, len(conversations_df))
        batch_df = conversations_df.iloc[start_idx:end_idx]
        
        print(f"Processing batch {batch_num + 1}/{total_batches} (rows {start_idx}-{end_idx-1})")
        
        for idx, row in batch_df.iterrows():
            try:
                # Add delay between each evaluation
                await asyncio.sleep(5)  # Increased delay for multiple metrics
                
                # Evaluate with Relevance
                relevance_result = relevance_eval(
                    query=row['user_prompt'],
                    response=row['assistant_response']
                )
                
                # Small delay between metrics
                await asyncio.sleep(2)
                
                # Evaluate with Coherence
                coherence_result = coherence_eval(
                    query=row['user_prompt'],
                    response=row['assistant_response']
                )
                
                # Small delay between metrics
                await asyncio.sleep(2)
                
                # Evaluate with Fluency
                fluency_result = fluency_eval(
                    query=row['user_prompt'],
                    response=row['assistant_response']
                )
                
                evaluation_results.append({
                    'brand_name': row['brand_name'],
                    'user_prompt': row['user_prompt'],
                    'assistant_response': row['assistant_response'],
                    'relevance_score': relevance_result['relevance'],
                    'coherence_score': coherence_result['coherence'],
                    'fluency_score': fluency_result['fluency'],
                    'turn_number': idx // 4 + 1
                })
                
                print(f"  Evaluated row {idx}: relevance={relevance_result['relevance']}, coherence={coherence_result['coherence']}, fluency={fluency_result['fluency']}")
                
            except Exception as e:
                print(f"Error evaluating conversation {idx}: {str(e)}")
                continue
        
        # Wait between batches (except for the last batch)
        if batch_num < total_batches - 1:
            print(f"Waiting {delay_between_batches} seconds before next batch...")
            await asyncio.sleep(delay_between_batches)
            
    return pd.DataFrame(evaluation_results)

In [16]:
evaluation_results_df = await evaluate_conversations_multi_metrics(conversations_df, batch_size=2,
    delay_between_batches=30)

if not evaluation_results_df.empty:
    print("\nEvaluation Results Summary:")
    print("Average Scores by Brand:")
    summary_stats = evaluation_results_df.groupby('brand_name')[['relevance_score', 'coherence_score', 'fluency_score']].mean()
    print(summary_stats.round(2))
    
    
    #evaluation_results_df.to_csv(file_path+'/conversation_metric_scores.csv', index=False)
    #summary_stats.to_csv(file_path+'/average_scores_by_brand.csv', index=True)
    print("\nResults saved to: evaluation_results/conversation_multi_metric_scores.csv")
else:
    print("No results generated")

Processing batch 1/5 (rows 0-1)
  Evaluated row 0: relevance=1.0, coherence=5.0, fluency=4.0
  Evaluated row 1: relevance=4.0, coherence=4.0, fluency=4.0
Waiting 30 seconds before next batch...
Processing batch 2/5 (rows 2-3)
  Evaluated row 2: relevance=5.0, coherence=4.0, fluency=4.0
  Evaluated row 3: relevance=4.0, coherence=4.0, fluency=4.0
Waiting 30 seconds before next batch...
Processing batch 3/5 (rows 4-5)
  Evaluated row 4: relevance=5.0, coherence=4.0, fluency=4.0
  Evaluated row 5: relevance=5.0, coherence=4.0, fluency=4.0
Waiting 30 seconds before next batch...
Processing batch 4/5 (rows 6-7)
  Evaluated row 6: relevance=5.0, coherence=4.0, fluency=4.0
  Evaluated row 7: relevance=4.0, coherence=4.0, fluency=4.0
Waiting 30 seconds before next batch...
Processing batch 5/5 (rows 8-9)
  Evaluated row 8: relevance=4.0, coherence=4.0, fluency=4.0
  Evaluated row 9: relevance=5.0, coherence=5.0, fluency=4.0

Evaluation Results Summary:
Average Scores by Brand:
                

In [26]:
import pandas as pd
file_path = 'C:/Users/t-toluale/.vscode/Ordering_ChatBot/evaluation_results/evaluation_1/evaluation_scores.csv'
evaluation_results_df = pd.read_csv(file_path)

In [27]:
import plotly.express as px
import matplotlib.pyplot as plt

# Create summary by brand
brand_summary = evaluation_results_df.groupby('brand_name')[['BrandVoiceMetric', 'RelevanceMetric', 'TaskCompletionMetric']].mean().reset_index()

# Multi-metric comparison by brand
fig1 = px.bar(
    brand_summary.melt(id_vars='brand_name', var_name='metric', value_name='score'),
    x='brand_name',
    y='score',
    color='metric',
    barmode='group',
    title='Custom Evaluation Scores by Brand and Metric (temp:0.7, top_p:0.95)',
    width=800,  
    height=500  
)
fig1.update_layout(legend_title_text='custom_metric')  
# Save as HTML (this always works)
fig1.write_html('C:/Users/t-toluale/.vscode/Ordering_ChatBot/evaluation_results/evaluation_1/custom_brand_metric_comparison.html')
#fig1.write_image('C:/Users/t-toluale/.vscode/Ordering_ChatBot/evaluation_results/evaluation_1/brand_metric_comparison.png')
fig1.show()

In [ ]:
fig1 = px.bar( evaluation_results_df.groupby('brand_name')['relevance_score'].mean().reset_index(), 
              x='brand_name', y='relevance_score', 
              title='Average Relevance Score by Brand' ) 
fig1.show()

In [ ]:
fig3 = px.line( evaluation_results_df.groupby(['brand_name', 'turn_number'])['relevance_score'].mean().reset_index(), 
               x='turn_number', y='relevance_score', color='brand_name', 
               title='Relevance Scores Over Conversation Progress' ) 
fig3.show()

In [ ]:
import json
from pathlib import Path

# Load the end-to-end evaluation results
results_file = 'C:/Users/t-toluale/.vscode/Ordering_ChatBot/evaluation_results/overall_evaluation_results/evaluation_results_o4-mini_20250729_220127.json'
with open(results_file, 'r') as f:
    data = json.load(f)

# Handle different JSON structures
if isinstance(data, list):
    conversations = data
elif isinstance(data, dict) and 'conversations' in data:
    conversations = data['conversations']
else:
    raise ValueError("Unexpected JSON structure. Expected either a list or a dict with 'conversations' key.")
total_conversations = len(conversations)

# Initialize metric accumulators
metrics_totals = {
    'groundedness': 0,
    'relevance': 0,
    'coherence': 0,
    'fluency': 0
}

# Collect all scores for detailed analysis
all_scores = {
    'groundedness': [],
    'relevance': [],
    'coherence': [],
    'fluency': []
}

print("END-TO-END EVALUATION METRICS ANALYSIS")
print("=====================================")
print(f"Total Conversations Analyzed: {total_conversations}")
print()

# Process each conversation
for i, conversation in enumerate(conversations, 1):
    query = conversation['original_data']['query']
    metrics = conversation['evaluation']
    
    print(f"Conversation {i}: \"{query[:50]}{'...' if len(query) > 50 else ''}\"")
    
    # Extract scores for each metric
    for metric_name in metrics_totals.keys():
        score = metrics[metric_name]['score']
        reason = metrics[metric_name]['reasoning']
        
        metrics_totals[metric_name] += score
        all_scores[metric_name].append(score)
        
        print(f"  {metric_name.capitalize()}: {score}/5 - {reason[:80]}{'...' if len(reason) > 80 else ''}")
    
    print()

# Calculate averages
averages = {}
for metric_name, total in metrics_totals.items():
    averages[metric_name] = total / total_conversations

print("AVERAGE SCORES SUMMARY")
print("=====================")
for metric_name, avg_score in averages.items():
    scores = all_scores[metric_name]
    min_score = min(scores)
    max_score = max(scores)
    
    print(f"{metric_name.upper()}: {avg_score:.2f}/5.00")
    print(f"  Range: {min_score} - {max_score}")
    print(f"  Distribution: {[scores.count(i) for i in range(1, 6)]}")
    print()

# Overall performance analysis
overall_average = sum(averages.values()) / len(averages)
print(f"OVERALL AVERAGE ACROSS ALL METRICS: {overall_average:.2f}/5.00")
print()

# Performance categories
print("PERFORMANCE BREAKDOWN BY SCORE:")
print("===============================")
for score in range(5, 0, -1):
    count_by_metric = {}
    for metric_name in metrics_totals.keys():
        count_by_metric[metric_name] = all_scores[metric_name].count(score)
    
    total_count = sum(count_by_metric.values())
    percentage = (total_count / (total_conversations * 4)) * 100  # 4 metrics per conversation
    
    print(f"Score {score}/5: {total_count} instances ({percentage:.1f}%)")
    for metric_name, count in count_by_metric.items():
        if count > 0:
            print(f"  {metric_name.capitalize()}: {count}")
    print()

# Identify strengths and weaknesses
print("STRENGTHS AND AREAS FOR IMPROVEMENT:")
print("===================================")

# Sort metrics by average score
sorted_metrics = sorted(averages.items(), key=lambda x: x[1], reverse=True)

print("Strongest Metrics:")
strongest_metrics = sorted_metrics[:2]
for metric_name, avg_score in strongest_metrics:
    print(f"  {metric_name.capitalize()}: {avg_score:.2f}/5.00")

print("\nAreas for Improvement:")
weakest_metrics = sorted_metrics[-2:]
for metric_name, avg_score in weakest_metrics:
    print(f"  {metric_name.capitalize()}: {avg_score:.2f}/5.00")

# Analyze conversations with lowest scores
print("\nCONVERSATIONS NEEDING ATTENTION:")
print("===============================")

conversation_scores = []
for i, conversation in enumerate(conversations):
    metrics = conversation['evaluation']
    total_score = sum(metrics[metric]['score'] for metric in metrics_totals.keys())
    avg_score = total_score / len(metrics_totals)
    conversation_scores.append((i, conversation['original_data']['query'], avg_score, total_score))

# Sort by average score
conversation_scores.sort(key=lambda x: x[2])

print("Lowest Performing Conversations:")
for i, (idx, query, avg_score, total_score) in enumerate(conversation_scores[:3]):
    print(f"{i+1}. \"{query[:60]}{'...' if len(query) > 60 else ''}\"")
    print(f"   Average Score: {avg_score:.2f}/5.00 (Total: {total_score}/20)")
    
    # Show specific weak areas
    conv_metrics = conversations[idx]['evaluation']
    weak_metrics = [(metric, data['score']) for metric, data in conv_metrics.items() if data['score'] < 4]
    if weak_metrics:
        print(f"   Weak Areas: {', '.join([f'{metric}({score})' for metric, score in weak_metrics])}")
    print()


END-TO-END EVALUATION METRICS ANALYSIS
Total Conversations Analyzed: 6

Conversation 1: "What menu items do you have?"
  Groundedness: 5/5 - The response accurately lists all the menu items and toppings exactly as provide...
  Relevance: 5/5 - The user asked, "What menu items do you have?" and the response directly and com...
  Coherence: 5/5 - The information is presented in a clear, logical structure with headings and bul...
  Fluency: 5/5 - The language is natural, grammatically correct, and warm, fitting the brand tone...

Conversation 2: "I want to order a burger with fries"
  Groundedness: 3/5 - The response generally aligns with the provided menu (offering hamburger, cheese...
  Relevance: 5/5 - The response directly addresses the user’s intent to order a burger with fries b...
  Coherence: 4/5 - The response follows a clear and logical structure—acknowledging the order, aski...
  Fluency: 5/5 - The language is natural, warm, and grammatically correct, matching the desired f...


TypeError: 'float' object is not subscriptable

: 